# ⚽ Football Analysis AI (Full System) — Google Colab
### Pipeline Phân Tích Bóng Đá Toàn Diện:
- 🎯 **YOLOv8 Object Detection & ByteTrack** (Cầu thủ, Trọng tài, Bóng)
- ⚡ **Đo Tốc độ tức thời (km/h), Quãng đường (km) & Sprint Highlight**
- 📊 **Bảng Dashboard Thể lực & Cường độ Vận động (Physical Analytics)**
- 🗺️ **Bản đồ nhiệt di chuyển 2D (Positional Heatmap)**
- 🕸️ **Đồ thị Mạng lưới Chuyền bóng (Passing Network Graph)**
- 🚩 **Công nghệ Vạch kẻ Việt vị Bán Tự Động (SAOT) & Bảng VAR Graphic**

> ⚙️ **LƯU Ý QUAN TRỌNG:** Vào menu **`Runtime`** -> **`Change runtime type`** -> Chọn **`T4 GPU`** để chạy với tốc độ cao nhất!

## 1️⃣ Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU đang sử dụng: {torch.cuda.get_device_name(0)}")

## 2️⃣ Cài đặt thư viện & Chuẩn bị mã nguồn

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q ultralytics supervision transformers umap-learn gdown pandas matplotlib

In [ ]:
import os

# Kiểm tra nếu đã có thư mục dự án (ví dụ: upload zip lên Colab)
if not os.path.exists('examples/football/main.py'):
    if os.path.exists('/content/Football_analysis.zip'):
        !unzip -q /content/Football_analysis.zip
        %cd Football_analysis
    else:
        # Clone từ GitHub nếu đã push code
        !git clone https://github.com/khangnguyenthe18/football_analysis.git
        %cd football_analysis

!pip install -e .
print("✅ Chuẩn bị mã nguồn thành công!")

## 3️⃣ Tải Model Weights & Video Mẫu từ Google Drive

In [ ]:
import os
DATA_DIR = "examples/football/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Tải 3 model weights
!gdown -O "{DATA_DIR}/football-ball-detection.pt" "https://drive.google.com/uc?id=1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V"
!gdown -O "{DATA_DIR}/football-player-detection.pt" "https://drive.google.com/uc?id=17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q"
!gdown -O "{DATA_DIR}/football-pitch-detection.pt" "https://drive.google.com/uc?id=1Ma5Kt86tgpdjCTKfum79YMgNnSjcoOyf"

# Tải 1 video mẫu (08fd33_0.mp4 ~ 20MB)
!gdown -O "{DATA_DIR}/08fd33_0.mp4" "https://drive.google.com/uc?id=1OG8K6wqUw9t7lp9ms1M48DxRhwTYciK-"

print("\n✅ Tải xong models và video!")
!ls -lh {DATA_DIR}/

## 4️⃣ Chạy Phân Tích Toàn Diện trên GPU (CUDA) 🚀

Lệnh dưới đây sẽ chạy toàn bộ các tính năng mới:
- Đo tốc độ tức thời ($km/h$) & vẽ nhãn trực tiếp trên cầu thủ
- Kẻ đường việt vị laser 3D & hiển thị bảng thông báo VAR
- Nhận diện đường chuyền, vệt bóng & HUD tỉ lệ kiểm soát bóng
- Xuất video thành phẩm & toàn bộ các biểu đồ phân tích vào thư mục `reports/`

In [ ]:
!python examples/football/main.py \
    --source_video_path examples/football/data/08fd33_0.mp4 \
    --target_video_path examples/football/data/08fd33_0-analysis.mp4 \
    --device cuda \
    --mode PASS_DETECTION \
    --reports_dir examples/football/reports/

## 5️⃣ Xem Dashboard Báo Cáo Thể Lực & Chiến Thuật Trực Tiếp

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

REP_DIR = "examples/football/reports"

# 1. Dashboard Thể Lực & Tốc Độ Cầu Thủ
dash_path = f"{REP_DIR}/physical_dashboard.png"
if os.path.exists(dash_path):
    plt.figure(figsize=(16, 9))
    plt.imshow(Image.open(dash_path))
    plt.axis('off')
    plt.title('PHYSICAL & SPEED PERFORMANCE DASHBOARD', fontsize=16, fontweight='bold')
    plt.show()

# 2. Bản Đồ Nhiệt Di Chuyển 2D (Heatmaps)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
hm1_path = f"{REP_DIR}/team_1_heatmap.png"
hm2_path = f"{REP_DIR}/team_2_heatmap.png"
if os.path.exists(hm1_path):
    axes[0].imshow(Image.open(hm1_path))
    axes[0].axis('off')
    axes[0].set_title('Team 1 - Positional Heatmap', fontsize=14, fontweight='bold')
if os.path.exists(hm2_path):
    axes[1].imshow(Image.open(hm2_path))
    axes[1].axis('off')
    axes[1].set_title('Team 2 - Positional Heatmap', fontsize=14, fontweight='bold')
plt.show()

# 3. Đồ Thị Mạng Lưới Chuyền Bóng (Passing Networks)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
net1_path = f"{REP_DIR}/team_1_passing_network.png"
net2_path = f"{REP_DIR}/team_2_passing_network.png"
if os.path.exists(net1_path):
    axes[0].imshow(Image.open(net1_path))
    axes[0].axis('off')
    axes[0].set_title('Team 1 - Passing Network', fontsize=14, fontweight='bold')
if os.path.exists(net2_path):
    axes[1].imshow(Image.open(net2_path))
    axes[1].axis('off')
    axes[1].set_title('Team 2 - Passing Network', fontsize=14, fontweight='bold')
plt.show()

# 4. Bảng Số Liệu Chi Tiết Thể Lực Cầu Thủ (CSV)
csv_path = f"{REP_DIR}/physical_stats.csv"
if os.path.exists(csv_path):
    df_phys = pd.read_csv(csv_path)
    print("\n📊 BẢNG THỐNG KÊ THỂ LỰC & TỐC ĐỘ:")
    display(df_phys)

## 6️⃣ Xem Video Thành Phẩm Có Vạch Việt Vị VAR & Tốc Độ Cầu Thủ

In [ ]:
import subprocess
from IPython.display import HTML
from base64 import b64encode

raw_video = "examples/football/data/08fd33_0-analysis.mp4"
web_video = "examples/football/data/08fd33_0-analysis-web.mp4"

if os.path.exists(raw_video):
    print("⏳ Đang nén video sang chuẩn HTML5 H.264 để phát mượt mà trên trình duyệt...")
    subprocess.run([
        "ffmpeg", "-y", "-i", raw_video,
        "-vcodec", "libx264", "-pix_fmt", "yuv420p",
        "-crf", "23", "-preset", "fast",
        web_video
    ], capture_output=True)

    target_file = web_video if os.path.exists(web_video) else raw_video
    mp4 = open(target_file, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=850 controls autoplay muted style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.5);">
        <source src="{data_url}" type="video/mp4">
    </video>
    """))